In [5]:
import torch

state_dict = torch.load("ulcer_model_state.pth", map_location="cpu")

In [6]:
# model architecture

from ulcer_model import UlcerNN

model = UlcerNN()
model.load_state_dict(state_dict)

<All keys matched successfully>

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report
import os

# Load test data from archive folder
test_dir = "./archive/test"  # Adjust path if needed

# Define transforms
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], 
                        std=[0.5, 0.5, 0.5])
])

# Create test dataset and loader
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize model evaluation
loss_function = nn.CrossEntropyLoss()
all_preds = []
all_labels = []
valid_loss = 0.0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        loss = loss_function(output, labels)
        valid_loss += loss.item()

        _, predictions = torch.max(output, 1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print results
print(f"Average Test Loss: {valid_loss / len(test_loader):.4f}\n")
print(classification_report(all_labels, all_preds, 
                           target_names=['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']))

Average Test Loss: 5.6906

              precision    recall  f1-score   support

     Grade 1       0.46      0.36      0.41        44
     Grade 2       0.33      0.32      0.33        37
     Grade 3       0.28      0.56      0.37        18
     Grade 4       0.65      0.52      0.58        42

    accuracy                           0.43       141
   macro avg       0.43      0.44      0.42       141
weighted avg       0.46      0.43      0.43       141

